In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn

In [2]:
import pandas as pd

matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

matches.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [3]:
matches = matches.dropna()

# Keep only useful columns
matches = matches[['team1','team2','toss_winner','toss_decision','winner','venue']]

matches.head()

,team1,team2,toss_winner,toss_decision,winner,venue
38,Delhi Daredevils,Kings XI Punjab,Delhi Daredevils,bat,Kings XI Punjab,Feroz Shah Kotla
41,Kolkata Knight Riders,Chennai Super Kings,Kolkata Knight Riders,bat,Chennai Super Kings,Eden Gardens
60,Delhi Daredevils,Kings XI Punjab,Delhi Daredevils,field,Delhi Daredevils,Newlands
63,Kings XI Punjab,Kolkata Knight Riders,Kolkata Knight Riders,field,Kolkata Knight Riders,Kingsmead
89,Chennai Super Kings,Kings XI Punjab,Chennai Super Kings,bat,Chennai Super Kings,SuperSport Park


In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in ['team1','team2','toss_winner','toss_decision','venue','winner']:
    matches[col] = le.fit_transform(matches[col])

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = matches.drop('winner', axis=1)
y = matches['winner']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print("Accuracy:", accuracy)

Accuracy: 0.0


In [6]:
def predict_winner(team1, team2, toss_winner, toss_decision, venue):
    input_data = [[team1, team2, toss_winner, toss_decision, venue]]
    return model.predict(input_data)

In [7]:
# Toss win impact
toss_win_match_win = (matches['toss_winner'] == matches['winner']).mean()
print("Toss Winner also won match:", toss_win_match_win)

# Bat vs Field
bat_first_win = matches[matches['toss_decision']==0]['winner'].count()
field_first_win = matches[matches['toss_decision']==1]['winner'].count()

print("Bat First Wins:", bat_first_win)
print("Field First Wins:", field_first_win)

Toss Winner also won match: 0.38095238095238093
Bat First Wins: 6
Field First Wins: 15


In [8]:
def calculate_cdi(runs_needed, overs_left, wickets_left):
    
    if overs_left == 0:
        return 100

    # Run Rate Factor
    rrr = runs_needed / overs_left
    run_rate_factor = (rrr / 8.5) * 50

    # Wicket Factor
    wicket_factor = ((10 - wickets_left) / 10) * 30

    # Phase Factor
    if overs_left > 10:
        phase_factor = 10
    elif overs_left > 5:
        phase_factor = 20
    else:
        phase_factor = 30

    # Final Score
    score = 0.5 * run_rate_factor + 0.3 * wicket_factor + 0.2 * phase_factor

    return min(100, round(score, 2))

In [9]:
score = calculate_cdi(runs_needed=80, overs_left=6, wickets_left=5)
print("Chasing Difficulty Index:", score)

Chasing Difficulty Index: 47.72


In [10]:
def difficulty_label(score):
    if score < 30:
        return "Easy 😌"
    elif score < 60:
        return "Moderate 😐"
    elif score < 80:
        return "Hard 😬"
    else:
        return "Very Hard 💀"